# Destination-choice + stay/move model via torch-choice

torch-choice port of `us/modeling_mnl.ipynb` (the Biogeme model). Same choice structure as the
Biogeme notebook: `alt=0` is staying, `alt=1..50` are the move alternatives (one row per person
per alternative in the reshaped data), and every person's utility is `V[0]` (stay) or `V[i]`
(move to alternative `i`), matching the Biogeme notebook's `V` dictionary.

**Coefficient sharing is preserved.** A number of Biogeme `Beta`s (e.g. `c_proportion_same_age_18_34`)
are used in *both* `V[0]` and `V[i]` with different variable expressions on each side (comparing the
person to their origin vs. to the candidate destination) but the *same* estimated coefficient. torch-choice
has no direct equivalent of Biogeme's named, reusable `Beta` objects, so this is reproduced by giving the
stay-context and move-context expressions for each shared concept the same column name in the reshaped
long-format table below (e.g. `proportion_same_age_18_34`) — since torch-choice's `ConditionalLogitModel`
assigns exactly one coefficient per input column, this ties the two contexts to one shared parameter,
exactly like the shared `Beta` does in Biogeme.

**Deviations from `modeling_mnl.ipynb` as currently written**, found while porting (the source notebook
does not currently run against the current `us_estdata_2018.parquet` schema either — these look like a
still-in-progress rename, not settled model changes):
- Cell 5's `ALT_VARYING_SUFFIXES` there still lists `OWN_RACE_PCT` / `OWN_GROUP_PCT`, which no longer exist
  in the parquet schema (they were renamed `OWN_RACE_ETH_PCT` / `OWN_NAICS_GROUP_PCT` — the names its own
  `V[i]` utility formulas in cell 23 already use). This port reads the new names directly.
- Cell 23 has `Variable(f"{alt}OWN_GROUPOWN_NAICS_GROUP_PCT_PCT")` for `c_proportion_same_naics_goods_trade`
  — a mangled column name (looks like a partially-applied find/replace). Treated here as
  `f"{alt}OWN_NAICS_GROUP_PCT"`, matching the other four `same_naics_*` terms in the same block.
- `c_proportion_same_race_white` and `c_proportion_same_race_other` reference columns that are not in
  `INDIV_COLS` at all (`WHITE`, `"Proportion of people White.ORIG"`), and the `V[i]` copy of the white-race
  term isn't even alternative-varying (no `{alt}` prefix — identical expression to `V[0]`'s). The `V[0]`
  version of the "other race" term also references a literally truncated column name (`"Proportion of "`).
  These three terms look like leftovers from the in-progress race/ethnicity recategorization (see the
  `add more comprehensive categorization of race/ethnicity, rename` commit) and are **omitted** here rather
  than guessed at — see the note near the shared-terms cell below.
- `ALT{i}_STATE` (and `CHOSEN` / `ALT{i}_PUMA`) are `string`-dtype in the current parquet, not
  `category`/`object`, so Biogeme's `df_train.select_dtypes(["number"])` (cell 9) would silently drop
  `STATE` entirely, and `Variable(f"{alt}STATE")` in cell 23 would then fail at estimation time. Since this
  port builds its own tensors instead of a Biogeme `Database`, `STATE` is just cast to `int` directly.
- The stay-side log-population offset (`log(Total Population....ORIG)`, the origin's population) is
  included alongside the move-side one (`log(ALT{i}_TOT_POP)`, the destination's population) — both are
  additive terms with an implicit coefficient of 1 (not estimated), matching Biogeme's un-parameterized
  `log(...)` calls in `V[0]` and `V[i]`.

**Optimizer note:** the `torch-choice` version pinned in this project (`1.0.6`) only exposes
`{SGD, Adagrad, Adadelta, Adam}` via `torch_choice.utils.run_helper.run` — no BFGS/LBFGS like Biogeme
uses, and no `.fit()`/device-aware API (that landed in `1.0.7`, which currently can't be installed here:
it pins `numpy<2.0`, conflicting with this project's `numpy 2.5.1`). Adam is used below; full-batch,
first-order optimization on a model this size will need more epochs and more tuning to converge than
Biogeme's BFGS does.


In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import torch
from torch_choice.data import ChoiceDataset
from torch_choice.model import ConditionalLogitModel
from torch_choice.utils.run_helper import run as tc_run

sys.path.insert(0, os.path.abspath(".."))
from lib import io as lio
from lib import util as lut


In [ ]:
year = 2018
num_alternatives = 50


### Read data

In [ ]:
df_train = lio.read_estdata(
    year=year,
    num_alternatives=num_alternatives,
)
print(df_train.shape)


### Reshape to long format, build torch-choice tensors

`lib.util.build_long_data` builds the long `(person_id, alt)` table shared by this notebook and
`modeling_larch.ipynb`: `alt=0` is staying, `alt=1..num_alternatives` are the move alternatives, with the
stay/move-context values for shared coefficients (e.g. `proportion_same_age_18_34`) written under the
same column name so they tie to one coefficient downstream, and `log_pop_offset` (destination
population on move rows, origin population on the stay row) left as an un-parameterized term. See its
docstring for the full column-by-column breakdown, including the two `modeling_mnl.ipynb` race terms
omitted for the same reasons noted in this notebook's intro cell.

The returned `long_df` is already sorted by `(person_id, alt)`, so each person occupies a contiguous
run of `num_alts` rows in a fixed alt order and the covariate matrix reshapes directly into
torch-choice's `(num_sessions, num_items, num_params)` layout without needing `EasyDatasetWrapper`'s
(much slower, per-column `pivot`-based) reshape.


In [ ]:
long_df, STAY_ONLY_TERMS, SHARED_TERMS, MOVE_ONLY_TERMS = lut.build_long_data(
    df_train, num_alternatives
)
varnames = STAY_ONLY_TERMS + SHARED_TERMS + MOVE_ONLY_TERMS

num_persons = df_train["person_id"].nunique()
num_alts = num_alternatives + 1  # 51: alt=0 (stay) + alt=1..50 (move)

X = (
    long_df[varnames]
    .to_numpy(dtype=np.float32)
    .reshape(num_persons, num_alts, len(varnames))
)
offset = (
    long_df["log_pop_offset"].to_numpy(dtype=np.float32).reshape(num_persons, num_alts)
)
choice_2d = long_df["choice"].to_numpy().reshape(num_persons, num_alts)
assert (choice_2d.sum(axis=1) == 1).all(), (
    "each person must choose exactly one alternative"
)
item_index = torch.from_numpy(choice_2d.argmax(axis=1)).long()

itemsession_x = torch.from_numpy(X)
offset_tensor = torch.from_numpy(offset)

dataset = ChoiceDataset(
    item_index=item_index,
    num_items=num_alts,
    num_sessions=num_persons,
    itemsession_x=itemsession_x,
)
dataset


### `OffsetConditionalLogitModel`: adding the un-parameterized `log(population)` term

`ConditionalLogitModel` assigns exactly one estimated coefficient per input column -- there's no way to
add a column to utility with a coefficient pinned at `1` (not trained), which is what Biogeme's bare
`log(Variable(...))` calls in `V[0]`/`V[i]` do (and what xlogit's `addit=` argument does). This subclass
adds `log_pop_offset` to the utility after the model's usual coefficient-weighted terms, as a registered
buffer: it moves with `.to(device)`, and being a buffer rather than a parameter, it's excluded from both
training (`.parameters()`) and the coefficient report.


In [ ]:
class OffsetConditionalLogitModel(ConditionalLogitModel):
    def __init__(self, *args, offset: torch.Tensor, **kwargs):
        super().__init__(*args, **kwargs)
        self.register_buffer("_offset", offset)

    def forward(self, batch, manual_coef_value_dict=None):
        total_utility = super().forward(batch, manual_coef_value_dict)
        return total_utility + self._offset[batch.session_index]


### Fitting

One coefficient per `varnames` entry (`"itemsession_x": "constant"`), no `intercept` key -- `stay` in
`STAY_ONLY_TERMS` is already an explicit ASC for staying, matching `fit_intercept=False` in the earlier
xlogit port / Biogeme not adding an implicit ASC of its own.


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

model = OffsetConditionalLogitModel(
    coef_variation_dict={"itemsession_x": "constant"},
    num_param_dict={"itemsession_x": len(varnames)},
    num_items=num_alts,
    offset=offset_tensor,
).to(device)
dataset = dataset.to(device)


In [ ]:
# NOTE: Adam, not BFGS/LBFGS -- see the optimizer note in the intro markdown cell. Full-batch
# (batch_size=-1) to match Biogeme/xlogit's full-batch MLE convention. Tune num_epochs/learning_rate;
# this is a starting point, not a verified-converged setting.
trained_model = tc_run(
    model,
    dataset,
    batch_size=-1,
    num_epochs=1000,
    learning_rate=0.05,
    model_optimizer="Adam",
    report_frequency=25,
    compute_std=True,
)


### Readable coefficient table

`run()`'s printed report names coefficients generically (`itemsession_x[constant]_0`, `_1`, ...) since
`ConditionalLogitModel` doesn't know about `varnames`. This re-attaches the actual variable names, in the
same order as `varnames`, for readability (matching Biogeme's/xlogit's named per-coefficient output) --
without standard errors, which would mean reimplementing `run()`'s internal Hessian computation with named
outputs; the printed report above already has those, just under the generic `_i` names.


In [ ]:
coef = trained_model.get_coefficient("itemsession_x[constant]").cpu().numpy()
pd.Series(coef, index=varnames, name="Estimation")
